In [31]:
import duckdb

In [23]:
con = duckdb.connect('md:')

con.execute("install duckpgq from community;")
con.execute("load duckpgq;")

con.execute("install ducklake;")
con.execute("load ducklake;")

con.sql(f"""
USE ducklake_bayesian;
""")

Attempting to automatically open the SSO authorization page in your default browser.
Please open this link to login into your account: https://auth.motherduck.com/activate?user_code=VMXF-JDPH


Token successfully retrieved ✅

You can display the token and store it as an environment variable to avoid having to log in again:
  PRAGMA PRINT_MD_TOKEN;


In [24]:
con.sql("DROP PROPERTY GRAPH IF EXISTS bayesian_kg")

con.sql("""
CREATE PROPERTY GRAPH bayesian_kg
  VERTEX TABLES (
    Factor, Disease, Symptom
  )
EDGE TABLES (
  CAUSES 	SOURCE KEY ('from_id') REFERENCES Factor (id)
                DESTINATION KEY ('to_id') REFERENCES Disease (id)
  LABEL CAUSES,
        
  COOCCURS SOURCE KEY ('from_id') REFERENCES Disease (id)
          DESTINATION KEY ('to_id') REFERENCES Disease (id)
  LABEL COOCCURS,
  
  DISEASE_LEADS_TO_FACTOR SOURCE KEY ('from_id') REFERENCES Disease (id)
          DESTINATION KEY ('to_id') REFERENCES Factor (id)
  LABEL DISEASE_LEADS_TO_FACTOR,
        
  HAS_SYMPTOM SOURCE KEY ('from_id') REFERENCES Disease (id)
          DESTINATION KEY ('to_id') REFERENCES Symptom (id)
  LABEL HAS_SYMPTOM,
  
  LEADS_TO_FACTOR SOURCE KEY ('from_id') REFERENCES Factor (id)
          DESTINATION KEY ('to_id') REFERENCES Factor (id)
  LABEL LEADS_TO_FACTOR,

  SYMPTOM_COOCCURS SOURCE KEY ('from_id') REFERENCES Symptom (id)
          DESTINATION KEY ('to_id') REFERENCES Symptom (id)
  LABEL SYMPTOM_COOCCURS     
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ 0 rows  │
└─────────┘

In [25]:
con.sql("""SELECT DISTINCT factor_name
  FROM GRAPH_TABLE (bayesian_kg
  MATCH
  (i:Factor)-[m:CAUSES]->(c:Disease WHERE c.name = 'COVID')
  COLUMNS (i.name AS factor_name)
  )
  LIMIT 30;
""")

┌─────────────┐
│ factor_name │
│   varchar   │
├─────────────┤
│ Smoking     │
│ Travel      │
└─────────────┘

In [26]:
con.close()

In [32]:
con = duckdb.connect('central.duckdb')

con.execute("install duckpgq from community;")
con.execute("load duckpgq;")

In [33]:
con.sql("DROP PROPERTY GRAPH IF EXISTS bayesian_kg")

con.sql("""
CREATE PROPERTY GRAPH bayesian_kg
  VERTEX TABLES (
    Factor, Disease, Symptom
  )
EDGE TABLES (
  CAUSES 	SOURCE KEY ('from_id') REFERENCES Factor (id)
                DESTINATION KEY ('to_id') REFERENCES Disease (id)
  LABEL CAUSES,
        
  COOCCURS SOURCE KEY ('from_id') REFERENCES Disease (id)
          DESTINATION KEY ('to_id') REFERENCES Disease (id)
  LABEL COOCCURS,
  
  DISEASE_LEADS_TO_FACTOR SOURCE KEY ('from_id') REFERENCES Disease (id)
          DESTINATION KEY ('to_id') REFERENCES Factor (id)
  LABEL DISEASE_LEADS_TO_FACTOR,
        
  HAS_SYMPTOM SOURCE KEY ('from_id') REFERENCES Disease (id)
          DESTINATION KEY ('to_id') REFERENCES Symptom (id)
  LABEL HAS_SYMPTOM,
  
  LEADS_TO_FACTOR SOURCE KEY ('from_id') REFERENCES Factor (id)
          DESTINATION KEY ('to_id') REFERENCES Factor (id)
  LABEL LEADS_TO_FACTOR,

  SYMPTOM_COOCCURS SOURCE KEY ('from_id') REFERENCES Symptom (id)
          DESTINATION KEY ('to_id') REFERENCES Symptom (id)
  LABEL SYMPTOM_COOCCURS     
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ 0 rows  │
└─────────┘

In [34]:
con.sql("""SELECT DISTINCT factor_name
  FROM GRAPH_TABLE (bayesian_kg
  MATCH
  (i:Factor)-[m:CAUSES]->(c:Disease WHERE c.name = 'COVID')
  COLUMNS (i.name AS factor_name)
  )
  LIMIT 30;
""")

┌─────────────┐
│ factor_name │
│   varchar   │
├─────────────┤
│ Smoking     │
│ Travel      │
└─────────────┘

In [36]:
con.close()